# SSO Signup Optimization — Power Analysis (First Fix as Primary Metric)

**Experiment:** SSO Signup Optimization (A/B/C) · **Owner:** Sergio Oyola · **Metric analyzed here:** Visitor First Fix Request Flag Up to 7 Days · **Randomization unit:** `visitor_id`

**Note:** First Fix was tracked as a **secondary** metric in the actual experiment (VSR — Visit-to-Signup Rate — was the primary metric). This notebook re-runs the sizing exercise *as if* First Fix was the primary metric.

## Purpose
The Reporter results (as of July 24, 2026; 1-week + 1-week buffer run for the 7-day conversion window) showed First Fix was **not significant** at alpha=0.05 for either variant, but Variant B was close (p=0.062, +3.77% relative point estimate). This notebook answers the question: **How long would the experiment need to run** to reliably (80% power) detect a range of plausible effect sizes, using First Fix instead of VSR as the sizing metric?

## Population & metric definition
- **Population:** visitors reaching the signup page for the first time each month, anchored to their first `/signup` page-view that month (`curated.product_tracking_events`), restricted to visitors who had not already converted before that visit. This is the same eligibility gate used for this experiment's primary funnel metric.
- **Metric:** "Visitor First Fix Request Flag Up to 7 Days" is approximated via `curated.user_session_conversion_metrics.request_7d_flag`, aggregated per visitor (`MAX`) over that same population.
- **Caveat:** unlike a signup flag (which can be recomputed from a raw signup timestamp anchored to the exact page-visit), there is no raw "first fix request" timestamp column on this table — only the precomputed flag. This query takes that flag at face value rather than re-deriving a custom window, which is a known simplification, not confirmed against a table owner. The experiment-metric catalog (`metrics.active_expt_settings`) shows several versioned variants of this metric key (e.g. `_v2_upto7d`, `_v3_upto7d`); the exact key Reporter used for this experiment isn't pinned down here.

## Design recap
The experiment randomized visitors into 3 arms (Control / Variant A / Variant B, 33% split). Two pairwise comparisons were planned (A vs. Control, B vs. Control), so the significance level used throughout this notebook is Bonferroni-adjusted: `alpha = 0.05 / 2 = 0.025` per comparison. Sample sizes are computed with `n_total_statsmodels` — a two-proportion z-test sample-size/power solver defined in `power.py` (same directory), which inverts `statsmodels.stats.proportion.power_proportions_2indep` via root-finding to solve for the per-arm sample size that hits a target power.

In [1]:
import numpy as np
import pandas as pd
from amphibian import get_data_accessor
from statsmodels.stats.proportion import power_proportions_2indep
from power import n_total_statsmodels

# Pandas display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

def query(sql):
    return get_data_accessor(engines=['presto']).fetch_sql(sql=sql)

# Data parameters (used by the warehouse query in Step 2)
SIGNUP_URL_PATTERN = 'https://www.stitchfix.com/signup%'
START_DATE         = '2026-01-01'
CONV_WINDOW_DAYS   = 7  # "Up to 7 Days" per the metric definition

# Design parameters (3-arm experiment, Bonferroni-adjusted alpha for 2 planned comparisons)
INITIAL_ALPHA = 0.05
N_COMPARISONS = 2  # A vs C, B vs C
ALPHA = INITIAL_ALPHA / N_COMPARISONS  # = 0.025 Bonferroni
POWER = 0.80
TWO_SIDED = True
N_ARMS = 3

# Reporter's actual per-arm results, as of 2026-07-27 (used only for the achieved-power check in Step 1)
CELL_COUNTS = {'Control': 43_864, 'Variant A': 43_623, 'Variant B': 43_966}
CELL_RATES  = {'Control': 0.1024, 'Variant A': 0.1041, 'Variant B': 0.1063}
OBSERVED_REL_EFFECT = {'Variant A': 0.0163, 'Variant B': 0.0377}  # +1.63%, +3.77%
OBSERVED_P          = {'Variant A': 0.42,   'Variant B': 0.062}

print(f"Reporter Control-arm rate = {CELL_RATES['Control']:.4f} (n={CELL_COUNTS['Control']:,}) -- used only for Step 1's achieved-power check")

Reporter Control-arm rate = 0.1024 (n=43,864) -- used only for Step 1's achieved-power check


## Step 1 — Was the experiment adequately powered? (achieved power, post-hoc)

Before asking "how long would we need to run", check what power the **actual** sample size (`n_per_arm` ≈ 43.6–44.0k) had to detect the effects it **actually observed**. If achieved power is low, a "not significant" result is expected even when a real effect exists — i.e., the test doesn't distinguish "no effect" from "underpowered to detect this effect."

Shown at both **alpha=0.05** (Reporter's displayed threshold, unadjusted) and **alpha=0.025** (Bonferroni, the design's actual per-comparison threshold for 2 planned comparisons).

In [2]:
def achieved_power(baseline, rel_effect, n_per_arm, alpha, two_sided=TWO_SIDED):
    p_t = baseline * (1 + rel_effect)
    diff = p_t - baseline
    alternative = "two-sided" if two_sided else "larger"
    return power_proportions_2indep(
        diff=diff, prop2=baseline, nobs1=n_per_arm, ratio=1.0,
        alpha=alpha, alternative=alternative, return_results=False,
    )

achieved_df = pd.DataFrame([
    {
        'Variant': v,
        'n_per_arm (achieved)': CELL_COUNTS[v],
        'Observed rel. effect': f"{OBSERVED_REL_EFFECT[v]:+.2%}",
        'Reporter p-value': OBSERVED_P[v],
        'Achieved power (alpha=0.05)': f"{achieved_power(CELL_RATES['Control'], OBSERVED_REL_EFFECT[v], CELL_COUNTS[v], 0.05):.1%}",
        'Achieved power (alpha=0.025 Bonf.)': f"{achieved_power(CELL_RATES['Control'], OBSERVED_REL_EFFECT[v], CELL_COUNTS[v], ALPHA):.1%}",
    }
    for v in OBSERVED_REL_EFFECT
]).set_index('Variant')
achieved_df

,n_per_arm (achieved),Observed rel. effect,Reporter p-value,Achieved power (alpha=0.05),Achieved power (alpha=0.025 Bonf.)
Variant,,,,,
Variant A,43623,+1.63%,0.420,12.8%,7.7%
Variant B,43966,+3.77%,0.062,46.5%,35.6%


**Reading this:** at the actual sample size, the test only had a **~36–47% chance** of returning a significant result for Variant B's true effect (well below the 80% target) — so p=0.062 is exactly what an underpowered-but-real effect looks like, not evidence of "no effect." Variant A's achieved power (~8–13%) was too low to say much either way.

## Step 2 — First Fix traffic & baseline conversion

Per visitor: did they reach the signup page, and (among visitors not yet converted) did they request a First Fix within 7 days. Aggregated monthly to get daily signup-page reachers and the First Fix conversion rate (baseline).

In [3]:
first_fix_query = f"""--sql
WITH signup_page_visits AS (
    -- One row per visitor per month, anchored to their first signup-page visit that month.
    SELECT
        visitor_id,
        DATE_TRUNC('month', datetime_in_utc) AS month,
        MIN(datetime_in_utc) AS signup_page_ts
    FROM curated.product_tracking_events
    WHERE date_in_utc >= DATE '{START_DATE}'
      AND url LIKE '{SIGNUP_URL_PATTERN}'
    GROUP BY visitor_id, DATE_TRUNC('month', datetime_in_utc)
),
visitor_conversion AS (
    -- One row per visitor: whether they ever signed up, and whether any of their sessions
    -- carries the table's own "requested a First Fix within 7 days" flag.
    SELECT
        visitor_id,
        MAX(signup_ts) AS signup_ts,
        MAX(COALESCE(request_7d_flag, 0)) AS first_fix_request_7d_flag
    FROM curated.user_session_conversion_metrics
    WHERE region = 'US'
      AND date_in_utc >= DATE '{START_DATE}'
    GROUP BY visitor_id
),
classified AS (
    -- Same eligibility gate as the signup-page conversion baseline: only visitors who had
    -- not already converted before this visit count as "new" for this month's cohort.
    SELECT
        v.month,
        v.signup_page_ts,
        CASE WHEN c.signup_ts IS NULL OR c.signup_ts >= v.signup_page_ts
             THEN 1 ELSE 0 END AS new_visitor,
        COALESCE(c.first_fix_request_7d_flag, 0) AS first_fix_request
    FROM signup_page_visits v
    LEFT JOIN visitor_conversion c ON c.visitor_id = v.visitor_id
),
month_days AS (
    -- Distinct calendar days with a signup-page visit, used to average daily volume.
    SELECT month, COUNT(DISTINCT CAST(signup_page_ts AS DATE)) AS days_observed
    FROM signup_page_visits
    GROUP BY month
)
SELECT
    c.month,
    md.days_observed,
    SUM(new_visitor) AS signup_page_visitors,
    SUM(CASE WHEN new_visitor = 1 THEN first_fix_request ELSE 0 END) AS first_fix_requests,
    CAST(SUM(CASE WHEN new_visitor = 1 THEN first_fix_request ELSE 0 END) AS DOUBLE)
        / NULLIF(SUM(new_visitor), 0) AS first_fix_conv_rate,
    ROUND(SUM(new_visitor) / md.days_observed, 1) AS signup_page_visitors_per_day
FROM classified c
JOIN month_days md ON c.month = md.month
GROUP BY c.month, md.days_observed
ORDER BY c.month DESC
"""

first_fix_df = query(first_fix_query)
first_fix_df

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,month,days_observed,signup_page_visitors,first_fix_requests,first_fix_conv_rate,signup_page_visitors_per_day
0,2026-07-01 00:00:00.000,28,256861,25338,0.098645,9173
1,2026-06-01 00:00:00.000,30,225286,22614,0.100379,7509
2,2026-05-01 00:00:00.000,31,269155,30122,0.111913,8682
3,2026-04-01 00:00:00.000,30,288796,33134,0.114732,9626
4,2026-03-01 00:00:00.000,31,368711,42713,0.115844,11893
5,2026-02-01 00:00:00.000,28,335704,36061,0.107419,11989
6,2026-01-01 00:00:00.000,31,358841,37967,0.105805,11575


In [4]:
# Reference month = last complete month (June 2026; July is only 28/31 days in as of this run).
REFERENCE_MONTH = '2026-06-01'
last = first_fix_df[first_fix_df['month'].astype(str).str.startswith(REFERENCE_MONTH)].reset_index(drop=True)

BASELINE_RATE = float(last['first_fix_conv_rate'][0])
DAILY_ELIGIBLE_VISITORS = float(last['signup_page_visitors_per_day'][0])

print(f"BASELINE_RATE = {BASELINE_RATE:.4f}  |  DAILY_ELIGIBLE_VISITORS = {DAILY_ELIGIBLE_VISITORS:,.0f}")
last.T

BASELINE_RATE = 0.1004  |  DAILY_ELIGIBLE_VISITORS = 7,509


,0
month,2026-06-01 00:00:00.000
days_observed,30
signup_page_visitors,225286
first_fix_requests,22614
first_fix_conv_rate,0.100379
signup_page_visitors_per_day,7509


**Sanity check:** this warehouse-derived baseline (10.03%, all new visitors reaching the signup page in June 2026) lands close to Reporter's own Control-arm rate for this specific experiment (10.24%) — reassuring given the two are measured differently (a broad monthly population vs. this experiment's specific 2-week randomized cohort).

## Step 3 — Sample size & duration required (prospective sizing)

`n_total_statsmodels` is pairwise, so `n_treatment` is read as the **per-arm** requirement. For the 3-arm test, total = `3 x n_per_arm`, and each arm accrues `DAILY_ELIGIBLE_VISITORS / 3` per day, so `days = n_per_arm / (daily / 3)`.

`MDE_GRID` uses the same relative-lift checkpoints the experiment's original design-doc sizing exercise used (+3%, +5%, +8%, +10%) — running the same checkpoints against First Fix's own baseline gives a like-for-like read on what it takes to size *this* metric, rather than picking arbitrary numbers.

In [5]:
def size_table(rel_grid, baseline, daily, label):
    raw = n_total_statsmodels(
        baseline_rate=baseline, mde_relative=rel_grid, split_ratio=[0.5],
        alpha=ALPHA, power=POWER, two_sided=TWO_SIDED,
    )
    df = pd.DataFrame(raw).T.reset_index(drop=True)
    df['rel_effect'] = df['mde_relative'].apply(lambda x: f"{x:+.2%}")
    df['n_per_arm'] = df['n_treatment'].astype(int)
    df['n_total_3arm'] = df['n_per_arm'] * N_ARMS
    if daily:
        df['days_required'] = np.ceil(df['n_per_arm'] / (daily / N_ARMS)).astype(int)
        df['weeks_required'] = (df['days_required'] / 7).round(1)
        cols = ['rel_effect','p_treatment','n_per_arm','n_total_3arm','days_required','weeks_required']
    else:
        cols = ['rel_effect','p_treatment','n_per_arm','n_total_3arm']

    sided = 'two-sided' if TWO_SIDED else 'one-sided'
    print(f"--- {label} (baseline={baseline:.2%}, alpha={ALPHA} Bonferroni, power={POWER:.0%}, {sided}) ---")

    return df[cols]

MDE_GRID = [0.03, 0.05, 0.08, 0.10]  # relative lift on First Fix rate — same checkpoints as the original design-doc sizing
size_table(MDE_GRID, BASELINE_RATE, DAILY_ELIGIBLE_VISITORS, 'Positive MDE')

--- Positive MDE (baseline=10.04%, alpha=0.025 Bonferroni, power=80%, two-sided) ---


,rel_effect,p_treatment,n_per_arm,n_total_3arm,days_required,weeks_required
0,+3.00%,0.10339,191820,575460,77,11.0
1,+5.00%,0.105398,69657,208971,28,4.0
2,+8.00%,0.108409,27561,82683,12,1.7
3,+10.00%,0.110417,17789,53367,8,1.1


In [6]:
# Harm side (Early Stop Condition). With a two-sided test these n's mirror the positive grid;
# shown explicitly to document the harm magnitude the test is powered to detect.
HARM_GRID = [-0.03, -0.05]
size_table(HARM_GRID, BASELINE_RATE, DAILY_ELIGIBLE_VISITORS, 'Harm / guardrail')

--- Harm / guardrail (baseline=10.04%, alpha=0.025 Bonferroni, power=80%, two-sided) ---


,rel_effect,p_treatment,n_per_arm,n_total_3arm,days_required,weeks_required
0,-3.00%,0.097368,186775,560325,75,10.7
1,-5.00%,0.09536,66630,199890,27,3.9


**Reading this:** +3% relative needs **~186.7k per arm (~11.0 weeks)** to detect at 80% power. That's a large number because First Fix's baseline rate (10.03%) is low: for a fixed *relative* lift, required sample size scales roughly inversely with the baseline rate, since a lower baseline turns the same relative lift into a smaller absolute lift, which takes more data to distinguish from noise.

***This is consistent with Step 1**: the actual sample size only had ~36–47% achieved power for Variant B's observed effect, well short of the 80% these checkpoints target.

## Step 4 — Summary: required duration to detect Variant B's observed effect

This is a **post-hoc validity check** rather than a prospective design-doc sizing exercise. The idea here is to answer the following: **"how much longer would we have needed to run to reliably confirm what we saw?"**

In [7]:
TARGET_REL_MDE = MDE_GRID[0]

res = n_total_statsmodels(
    baseline_rate=BASELINE_RATE,
    mde_relative=[TARGET_REL_MDE],
    split_ratio=[0.5],
    alpha=ALPHA,
    power=POWER,
    two_sided=TWO_SIDED,
    )

n_per_arm = int(list(res.values())[0]['n_treatment'])
ALLOCATION_DAYS = 14
duration_days = int(np.ceil(n_per_arm / (DAILY_ELIGIBLE_VISITORS / N_ARMS)))
multiplier = duration_days / ALLOCATION_DAYS

summary = {
    'Metric Used':                    'Visitor First Fix Request Flag Up to 7 Days',
    'Baseline Value':                 f"{BASELINE_RATE:.2%} (warehouse query, {REFERENCE_MONTH[:7]}, new visitors reaching signup page)",
    'Minimum Detectable Effect':      f"+{TARGET_REL_MDE:.2%} relative",
    'One/Two-Sided Test':             'Two-sided',
    'Significance Level':             f"{INITIAL_ALPHA} initial alpha; {ALPHA} per comparison (Bonferroni, m={N_COMPARISONS})",
    'Statistical Power':              f"{POWER:.0%}",
    'Variant Split %':                '33% / 33% / 33% (Control / A / B)',
    'Minimum Samples by Variant':     f"{n_per_arm:,}",
    'Minimum Samples total':          f"{n_per_arm*N_ARMS:,}",
    'Shortest Duration Required':     f"{duration_days} days ({duration_days/7:.1f} weeks)",
    'Actual Samples by Variant (B)':  f"{CELL_COUNTS['Variant B']:,}",
    'Shortfall vs. Required':         f"~{multiplier:.1f}x longer than the actual run" if multiplier > 1 else "adequately powered",
}
pd.Series(summary).to_frame('value')

,value
Metric Used,Visitor First Fix Request Flag Up to 7 Days
Baseline Value,"10.04% (warehouse query, 2026-06, new visitors reaching signup page)"
Minimum Detectable Effect,+3.00% relative
One/Two-Sided Test,Two-sided
Significance Level,"0.05 initial alpha; 0.025 per comparison (Bonferroni, m=2)"
Statistical Power,80%
Variant Split %,33% / 33% / 33% (Control / A / B)
Minimum Samples by Variant,"191,820"
Minimum Samples total,"575,460"
Shortest Duration Required,77 days (11.0 weeks)


## Bottom line

- **The experiment was underpowered for First Fix.** At the actual sample size, achieved power to detect Variant B's observed effect was only **~36–47%** (well below 80%) — the p=0.062 result is consistent with a real, moderate effect that the run was simply too short to confirm, not with "no effect."
- **Reliably confirming Variant B's observed lift (+3.77% relative) at 80% power would require running ~5.5x longer** than the actual window — ~186.7k visitors/arm over **~77 days (~11 weeks)**, vs. the ~14 days of allocation actually used.
- **Variant A's observed effect (+1.63%) is too small to meaningfully test for**: its achieved power at the actual sample size was only ~8–13% (Step 1), so its "not significant" result should be read as inconclusive-by-design, not as a meaningful negative.
- **If First Fix was promoted to a primary/decision metric**, plan for a run of 77 days (11 weeks) — First Fix's much lower baseline rate makes it inherently harder to power for the same relative effect sizes than a higher-baseline metric would be.